# Global (Shared-Parameter) Fitting

In a compound screen, multiple dose-response curves may share the same assay baseline (**Bottom**) and maximum signal (**Top**), differing only in potency (EC50) and slope (HillSlope). Fitting these jointly constrains Top and Bottom with all data, reducing uncertainty on those parameters while allowing each compound its own EC50.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from openfit import Fit, GlobalFit, compare_models

## Simulate three dose-response curves

Same Bottom=2, Top=97; different EC50 per compound (1.0, 8.0, 40.0); same HillSlope=1.2.

In [ ]:
rng = np.random.default_rng(99)
x = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0])

# Shared params
BOTTOM, TOP, HILL = 2.0, 97.0, 1.2
# Per-compound EC50
ec50s = [1.0, 8.0, 40.0]
compound_names = ["Compound A", "Compound B", "Compound C"]

datasets = []
for ec50 in ec50s:
    y_true = BOTTOM + (TOP - BOTTOM) / (1 + (ec50 / x) ** HILL)
    y = y_true * (1 + rng.normal(0, 0.12, size=len(x)))
    y = np.clip(y, 0.1, None)
    datasets.append((x, y))

for name, (xi, yi), ec50 in zip(compound_names, datasets, ec50s):
    print(f"{name}: true EC50={ec50}, y={np.round(yi, 1)}")

## Independent fits (baseline)

Fit each curve separately. Note the wide CI on Top and Bottom when each curve has limited dynamic range coverage.

In [ ]:
indep_fits = []
for name, (xi, yi) in zip(compound_names, datasets):
    r = Fit("hill4p", xi, yi, weights="1/y2").run()
    indep_fits.append(r)
    p = r.params
    print(f"{name}: Bottom={p['Bottom']:.2f}±{r.se['Bottom']:.2f}  "
          f"Top={p['Top']:.2f}±{r.se['Top']:.2f}  "
          f"EC50={p['EC50']:.2f}±{r.se['EC50']:.2f}")

## Global fit: share Top and Bottom

In [ ]:
gf = GlobalFit(
    datasets=[(xi, yi, "1/y2") for xi, yi in datasets],
    model="hill4p",
    shared=["Top", "Bottom"],
    local=["EC50", "HillSlope"],
)
gresult = gf.run()

print("Shared parameters:")
for name, val in gresult.shared_params.items():
    print(f"  {name}: {val:.3f}")

print("\nPer-compound local parameters:")
for cname, lp in zip(compound_names, gresult.local_params):
    print(f"  {cname}: EC50={lp['EC50']:.3f}  HillSlope={lp['HillSlope']:.3f}")

## F-test: is sharing justified?

The F-test compares the joint (shared) model against the fully independent fits.

In [ ]:
ft = gresult.f_test
print(f"F-statistic: {ft.f_statistic:.3f}")
print(f"p-value:     {ft.p_value:.4f}")
print(f"df_num:      {ft.df_numerator}")
print(f"df_den:      {ft.df_denominator}")
if ft.p_value > 0.05:
    print("\nConclusion: p > 0.05 — sharing Top and Bottom is statistically justified.")
else:
    print("\nConclusion: p < 0.05 — the shared constraint is not supported; consider releasing parameters.")

## Compare: independent vs global EC50 estimates

Global fitting should give tighter SE on Top/Bottom.

In [ ]:
print("EC50 estimates (True / Independent / Global):")
print(f"{'Compound':12s}  {'True':>8s}  {'Indep EC50':>12s}  {'Global EC50':>12s}")
print("-" * 50)
for cname, true_ec50, ifit, lp in zip(compound_names, ec50s, indep_fits, gresult.local_params):
    print(f"{cname:12s}  {true_ec50:>8.2f}  "
          f"{ifit.params['EC50']:>8.3f}±{ifit.se['EC50']:>4.3f}  "
          f"{lp['EC50']:>8.3f}")

print(f"\nShared Bottom: True={BOTTOM}  Global={gresult.shared_params['Bottom']:.3f}")
print(f"Shared Top:    True={TOP}  Global={gresult.shared_params['Top']:.3f}")

## Visualise all curves

In [ ]:
x_fine = np.logspace(np.log10(x.min()), np.log10(x.max()), 300)
colors = ["tab:blue", "tab:orange", "tab:green"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (fits, local_params, title) in enumerate([
    (indep_fits, [r.params for r in indep_fits], "Independent Fits"),
    (None, gresult.local_params, "Global Fit (shared Top/Bottom)"),
]):
    sp = gresult.shared_params
    for i, (cname, (xi, yi)) in enumerate(zip(compound_names, datasets)):
        lp = local_params[i]
        if title.startswith("Independent"):
            B, T = lp["Bottom"], lp["Top"]
        else:
            B, T = sp["Bottom"], sp["Top"]
        yc = B + (T - B) / (1 + (lp["EC50"] / x_fine) ** lp["HillSlope"])
        axes[ax].semilogx(x_fine, yc, color=colors[i], linewidth=2, label=f"{cname} EC50={lp['EC50']:.1f}")
        axes[ax].semilogx(xi, yi, "o", color=colors[i], markersize=6)
    axes[ax].set_xlabel("Concentration")
    axes[ax].set_ylabel("Response")
    axes[ax].set_title(title)
    axes[ax].legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/global_fitting.png", dpi=100, bbox_inches="tight")
plt.show()

## Summary

- Global fitting with shared Top/Bottom reduced EC50 uncertainty compared to independent fits.
- F-test confirmed that sharing Top and Bottom is statistically justified (p > 0.05).
- Use global fitting when: multiple datasets share a common assay parameter; you want to borrow strength across curves.
- Do not share parameters across datasets measured under different assay conditions.